In [1]:
import pandas as pd
import numpy as np
import joblib
import os

In [2]:
gps = pd.read_csv(
    "../data/processed/gps_anomalies.csv"
)

cases = pd.read_csv(
    "../data/synthetic/missing_person_cases.csv"
)

transition_matrix = pd.read_csv(
    "../data/processed/transition_matrix.csv",
    index_col=0
)

print("GPS:", gps.shape)
print("Cases:", cases.shape)
print("Transition matrix:", transition_matrix.shape)

GPS: (24730077, 23)
Cases: (100, 15)
Transition matrix: (10, 10)


In [3]:
rf_model = joblib.load(
    "../models/location_model.pkl"
)

target_encoder = joblib.load(
    "../models/target_encoder.pkl"
)

location_features = joblib.load(
    "../models/location_features.pkl"
)

print("Location model loaded!")

Location model loaded!


In [4]:
case = cases.iloc[0]

print("Case ID:", case["Case_ID"])
print("Age Group:", case["Age_Group"])
print("Last Latitude:", case["Last_Latitude"])
print("Last Longitude:", case["Last_Longitude"])
print("Last Seen Time:", case["Last_Seen_Time"])
print("Day:", case["Day"])

Case ID: MP-2026-001
Age Group: 36-45
Last Latitude: 39.980525
Last Longitude: 116.44944
Last Seen Time: 2008-10-12 08:38:24
Day: Sunday


In [5]:
case_input = pd.DataFrame(
    [case]
)

In [6]:
feature_columns = [
    "Age_Group",
    "Day",
    "Weather",
    "Last_Latitude",
    "Last_Longitude",
    "Average_Distance",
    "Average_Speed",
    "Previous_Area",
    "Usual_Area",
    "Time_Since_Last_Seen",
    "Hour"
]

In [7]:
case_input["Last_Seen_Time"] = pd.to_datetime(
    case_input["Last_Seen_Time"]
)

case_input["Hour"] = (
    case_input["Last_Seen_Time"].dt.hour
)

In [8]:
case_input = case_input[
    feature_columns
]

In [9]:
case_input = pd.get_dummies(
    case_input,
    columns=[
        "Age_Group",
        "Day",
        "Weather"
    ],
    dtype=int
)

In [10]:
case_input = case_input.reindex(
    columns=location_features,
    fill_value=0
)

print(
    "Input shape:",
    case_input.shape
)

Input shape: (1, 22)


In [11]:
case_probabilities = rf_model.predict_proba(
    case_input
)[0]

predicted_classes = rf_model.classes_

ml_scores = pd.DataFrame({
    "Area": predicted_classes,
    "ML_Probability": case_probabilities
})

ml_scores["Area"] = (
    target_encoder.inverse_transform(
        ml_scores["Area"].astype(int)
    )
)

ml_scores

,Area,ML_Probability
0,0,0.0
1,1,0.0
2,2,0.0
3,5,0.0
4,6,0.0
5,7,0.0
6,9,1.0


In [12]:
ml_scores["ML_Score"] = (
    ml_scores["ML_Probability"] * 100
)

ml_scores["ML_Score"] = (
    ml_scores["ML_Score"].round(2)
)

ml_scores

,Area,ML_Probability,ML_Score
0,0,0.0,0.0
1,1,0.0,0.0
2,2,0.0,0.0
3,5,0.0,0.0
4,6,0.0,0.0
5,7,0.0,0.0
6,9,1.0,100.0


In [13]:
valid_gps = gps[
    gps["cluster"] != -1
].copy()

historical_frequency = (
    valid_gps["cluster"]
    .value_counts()
    .reset_index()
)

historical_frequency.columns = [
    "Area",
    "Visit_Count"
]

historical_frequency

,Area,Visit_Count
0,9,14645638
1,1,5487320
2,2,1160492
3,0,785358
4,6,683587
5,4,591782
6,8,582360
7,7,324694
8,5,298423
9,3,170423


In [14]:
max_frequency = (
    historical_frequency["Visit_Count"]
    .max()
)

historical_frequency["Historical_Score"] = (
    historical_frequency["Visit_Count"]
    / max_frequency
) * 100

historical_frequency["Historical_Score"] = (
    historical_frequency["Historical_Score"]
    .round(2)
)

historical_frequency

,Area,Visit_Count,Historical_Score
0,9,14645638,100.00
1,1,5487320,37.47
2,2,1160492,7.92
3,0,785358,5.36
4,6,683587,4.67
5,4,591782,4.04
6,8,582360,3.98
7,7,324694,2.22
8,5,298423,2.04
9,3,170423,1.16


In [15]:
start_area = int(
    case["Previous_Area"]
)

print(
    "Starting Area:",
    start_area
)

Starting Area: 9


In [16]:
if str(start_area) in transition_matrix.index:
    
    route_scores = (
        transition_matrix
        .loc[str(start_area)]
        .reset_index()
    )
    
    route_scores.columns = [
        "Area",
        "Route_Probability"
    ]

else:
    
    route_scores = pd.DataFrame({
        "Area": [],
        "Route_Probability": []
    })

In [17]:
route_scores["Area"] = pd.to_numeric(
    route_scores["Area"],
    errors="coerce"
)

route_scores = route_scores.dropna(
    subset=["Area"]
)

route_scores["Route_Score"] = (
    route_scores["Route_Probability"] * 100
)

route_scores["Route_Score"] = (
    route_scores["Route_Score"].round(2)
)

route_scores.head()

,Area,Route_Probability,Route_Score


In [18]:
area_centers = (
    valid_gps
    .groupby("cluster")
    .agg(
        Area_Latitude=("latitude", "mean"),
        Area_Longitude=("longitude", "mean")
    )
    .reset_index()
)

area_centers.columns = [
    "Area",
    "Area_Latitude",
    "Area_Longitude"
]

area_centers

,Area,Area_Latitude,Area_Longitude
0,0,42.367534,117.556758
1,1,39.885085,116.514713
2,2,30.193778,114.729449
3,3,44.775164,6.829885
4,4,20.073562,111.095783
5,5,42.804797,-117.331026
6,6,38.561120,116.251663
7,7,35.948550,96.992804
8,8,34.994947,114.164560
9,9,40.009068,116.343671


In [19]:
def haversine_distance(
    lat1,
    lon1,
    lat2,
    lon2
):
    
    R = 6371
    
    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)
    
    dlat = lat2 - lat1
    dlon = np.radians(lon2) - np.radians(lon1)
    
    a = (
        np.sin(dlat / 2) ** 2
        +
        np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )
    
    c = 2 * np.arcsin(
        np.sqrt(a)
    )
    
    return R * c

In [20]:
area_centers["Distance_km"] = haversine_distance(
    case["Last_Latitude"],
    case["Last_Longitude"],
    area_centers["Area_Latitude"],
    area_centers["Area_Longitude"]
)

area_centers.head()

,Area,Area_Latitude,Area_Longitude,Distance_km
0,0,42.367534,117.556758,281.131562
1,1,39.885085,116.514713,11.983269
2,2,30.193778,114.729449,1099.364492
3,3,44.775164,6.829885,8266.387374
4,4,20.073562,111.095783,2271.569377


In [21]:
max_distance = (
    area_centers["Distance_km"]
    .max()
)

if max_distance > 0:
    
    area_centers["Distance_Score"] = (
        1
        -
        area_centers["Distance_km"]
        / max_distance
    ) * 100

else:
    
    area_centers["Distance_Score"] = 100

In [22]:
area_centers["Distance_Score"] = (
    area_centers["Distance_Score"]
    .clip(0, 100)
    .round(2)
)

area_centers[
    [
        "Area",
        "Distance_km",
        "Distance_Score"
    ]
]

,Area,Distance_km,Distance_Score
0,0,281.131562,96.99
1,1,11.983269,99.87
2,2,1099.364492,88.23
3,3,8266.387374,11.51
4,4,2271.569377,75.68
5,5,9341.107877,0.00
6,6,158.746110,98.30
7,7,1759.623113,81.16
8,8,589.827170,93.69
9,9,9.552758,99.90


In [23]:
gps["timestamp"] = pd.to_datetime(
    gps["timestamp"],
    errors="coerce"
)

gps["hour"] = (
    gps["timestamp"].dt.hour
)

In [24]:
case_time = pd.to_datetime(
    case["Last_Seen_Time"]
)

case_hour = case_time.hour

print(
    "Case hour:",
    case_hour
)

Case hour: 8


In [25]:
time_counts = (
    valid_gps[
        valid_gps["hour"] == case_hour
    ]
    .groupby("cluster")
    .size()
    .reset_index(
        name="Time_Count"
    )
)

time_counts.columns = [
    "Area",
    "Time_Count"
]

if len(time_counts) > 0:
    
    max_time = (
        time_counts["Time_Count"]
        .max()
    )
    
    time_counts["Time_Score"] = (
        time_counts["Time_Count"]
        / max_time
    ) * 100
    
else:
    
    time_counts["Time_Score"] = 0

time_counts["Time_Score"] = (
    time_counts["Time_Score"]
    .round(2)
)

time_counts

,Area,Time_Count,Time_Score
0,0,57668,7.01
1,1,396136,48.16
2,2,65894,8.01
3,3,6288,0.76
4,4,31999,3.89
5,5,4894,0.59
6,6,36952,4.49
7,7,16491,2.00
8,8,26387,3.21
9,9,822607,100.00


In [26]:
anomaly_data = valid_gps.copy()

anomaly_summary = (
    anomaly_data
    .groupby("cluster")
    .agg(
        Total_Points=(
            "anomaly",
            "count"
        ),
        Anomaly_Points=(
            "anomaly",
            lambda x: (x == -1).sum()
        )
    )
    .reset_index()
)

In [27]:
anomaly_summary["Anomaly_Rate"] = (
    anomaly_summary["Anomaly_Points"]
    /
    anomaly_summary["Total_Points"]
) * 100

anomaly_summary["Anomaly_Score"] = (
    anomaly_summary["Anomaly_Rate"]
    /
    anomaly_summary["Anomaly_Rate"].max()
) * 100

anomaly_summary["Anomaly_Score"] = (
    anomaly_summary["Anomaly_Score"]
    .fillna(0)
    .round(2)
)

anomaly_summary

,cluster,Total_Points,Anomaly_Points,Anomaly_Rate,Anomaly_Score
0,0,785358,18057,2.299206,5.77
1,1,5487320,9222,0.168060,0.42
2,2,1160492,30110,2.594589,6.52
3,3,170423,5559,3.261884,8.19
4,4,591782,30858,5.214420,13.10
5,5,298423,118826,39.817976,100.00
6,6,683587,11084,1.621447,4.07
7,7,324694,932,0.287039,0.72
8,8,582360,14414,2.475101,6.22
9,9,14645638,8230,0.056194,0.14


In [28]:
priority = ml_scores[
    [
        "Area",
        "ML_Score"
    ]
].copy()

In [29]:
priority = priority.merge(
    historical_frequency[
        [
            "Area",
            "Historical_Score"
        ]
    ],
    on="Area",
    how="left"
)

In [30]:
priority = priority.merge(
    route_scores[
        [
            "Area",
            "Route_Score"
        ]
    ],
    on="Area",
    how="left"
)

In [31]:
priority = priority.merge(
    area_centers[
        [
            "Area",
            "Distance_Score"
        ]
    ],
    on="Area",
    how="left"
)

In [32]:
priority = priority.merge(
    time_counts[
        [
            "Area",
            "Time_Score"
        ]
    ],
    on="Area",
    how="left"
)

In [33]:
priority = priority.merge(
    anomaly_summary[
        [
            "cluster",
            "Anomaly_Score"
        ]
    ].rename(
        columns={
            "cluster": "Area"
        }
    ),
    on="Area",
    how="left"
)

In [34]:
score_columns = [
    "ML_Score",
    "Historical_Score",
    "Route_Score",
    "Distance_Score",
    "Time_Score",
    "Anomaly_Score"
]

priority[score_columns] = (
    priority[score_columns]
    .fillna(0)
)

priority

,Area,ML_Score,Historical_Score,Route_Score,Distance_Score,Time_Score,Anomaly_Score
0,0,0.0,5.36,0.0,96.99,7.01,5.77
1,1,0.0,37.47,0.0,99.87,48.16,0.42
2,2,0.0,7.92,0.0,88.23,8.01,6.52
3,5,0.0,2.04,0.0,0.00,0.59,100.00
4,6,0.0,4.67,0.0,98.30,4.49,4.07
5,7,0.0,2.22,0.0,81.16,2.00,0.72
6,9,100.0,100.00,0.0,99.90,100.00,0.14


In [35]:
priority["Priority_Score"] = (
    
    priority["ML_Score"] * 0.30
    
    +
    
    priority["Historical_Score"] * 0.20
    
    +
    
    priority["Route_Score"] * 0.15
    
    +
    
    priority["Distance_Score"] * 0.15
    
    +
    
    priority["Time_Score"] * 0.10
    
    +
    
    priority["Anomaly_Score"] * 0.10
)

In [36]:
priority["Priority_Score"] = (
    priority["Priority_Score"]
    .clip(0, 100)
    .round(2)
)

In [37]:
def assign_priority(score):
    
    if score <= 30:
        return "Low"
    
    elif score <= 60:
        return "Medium"
    
    elif score <= 80:
        return "High"
    
    else:
        return "Very High"

In [38]:
priority["Priority"] = (
    priority["Priority_Score"]
    .apply(assign_priority)
)

In [39]:
priority = priority.sort_values(
    "Priority_Score",
    ascending=False
).reset_index(
    drop=True
)

priority["Rank"] = (
    priority.index + 1
)

In [40]:
final_priority = priority[
    [
        "Rank",
        "Area",
        "ML_Score",
        "Historical_Score",
        "Route_Score",
        "Distance_Score",
        "Time_Score",
        "Anomaly_Score",
        "Priority_Score",
        "Priority"
    ]
]

final_priority

,Rank,Area,ML_Score,Historical_Score,Route_Score,Distance_Score,Time_Score,Anomaly_Score,Priority_Score,Priority
0,1,9,100.0,100.00,0.0,99.90,100.00,0.14,75.00,High
1,2,1,0.0,37.47,0.0,99.87,48.16,0.42,27.33,Low
2,3,0,0.0,5.36,0.0,96.99,7.01,5.77,16.90,Low
3,4,6,0.0,4.67,0.0,98.30,4.49,4.07,16.54,Low
4,5,2,0.0,7.92,0.0,88.23,8.01,6.52,16.27,Low
5,6,7,0.0,2.22,0.0,81.16,2.00,0.72,12.89,Low
6,7,5,0.0,2.04,0.0,0.00,0.59,100.00,10.47,Low


In [41]:
os.makedirs(
    "../data/processed",
    exist_ok=True
)

final_priority.to_csv(
    "../data/processed/search_priority_scores.csv",
    index=False
)

print(
    "Search priority scores saved successfully!"
)

Search priority scores saved successfully!


In [42]:
print(
    "========== TOP 5 SEARCH PRIORITY AREAS =========="
)

display(
    final_priority.head(5)
)

========== TOP 5 SEARCH PRIORITY AREAS ==========


,Rank,Area,ML_Score,Historical_Score,Route_Score,Distance_Score,Time_Score,Anomaly_Score,Priority_Score,Priority
0,1,9,100.0,100.00,0.0,99.90,100.00,0.14,75.00,High
1,2,1,0.0,37.47,0.0,99.87,48.16,0.42,27.33,Low
2,3,0,0.0,5.36,0.0,96.99,7.01,5.77,16.90,Low
3,4,6,0.0,4.67,0.0,98.30,4.49,4.07,16.54,Low
4,5,2,0.0,7.92,0.0,88.23,8.01,6.52,16.27,Low


In [43]:
top5 = final_priority.head(5).copy()

top5.to_csv(
    "../data/processed/top5_search_areas.csv",
    index=False
)

print(
    "Top-5 search areas saved!"
)

Top-5 search areas saved!
